In [2]:
# print('king')

In [16]:
import torch

if torch.cuda.is_available():
    print("CUDA is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")


CUDA is available!
GPU Name: NVIDIA A100-SXM4-40GB


In [8]:
import os

dataset_path = '/kaggle/input/vehicle-orientation-dataset-part-1/vehicle-orientation-1'
all_files = os.listdir(dataset_path)
print(all_files[:20])  # show first 20 files


['02A2J14V8353J5RY4CQG.txt', '9VTATFEHET2JT91ZYSU9.jpg', '7RQVS0Z11OKDEPFLLM4E.jpg', 'X0WRQ1X5BIP2OGI2JG10.jpg', 'RGVJYH9HINPR49LA696D.jpg', 'VB389UFNOXBS34UBM4WE.txt', '6ILV08QDPRTTFELTE8LD.txt', 'L01BB0XE29VTVXVSRJRE.txt', 'MN3V9H3WUFJHTZVFACST.txt', 'YDHTO830SYGKDLXUBK7W.txt', 'XR54VX0ZM8I6LVS7TUQN.jpg', 'WLLF1VYWL0RQDHZTUX0X.jpg', '22MF57AQXY75IQJO9X5R.jpg', 'MEMTXNF237C0QUP520A3.txt', 'DGLIJWKKJZAGXTOZ6QV2.jpg', 'DNLZLQ1SX5CJHC8236SA.jpg', '2YG8GXUOQ3SYL7ZQML9J.txt', 'SN8BKVJYMZTTZMF1U93K.txt', 'TR2GRHJA9S1JAJ6RTL02.txt', 'TOPY2YYF2VZ536ENIIRM.txt']


In [9]:
import os

dataset_path = '/kaggle/working/'

# Find all image files
image_files = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(root, f))

# Count images
print(f"Total number of images: {len(image_files)}")


Total number of images: 30006


Total number of files: 0


In [ ]:
import os

# Create a folder
os.makedirs('/kaggle/working/my_folder', exist_ok=True)


In [ ]:
import os
import shutil

# Paths
input_folder = '/kaggle/input/vehicle-orientation-dataset-part-1/vehicle-orientation-1'
images_folder = '/home/tsawang/vehicle_images'
annotations_folder = '/home/tsawang/annotation'

# Create folders if they don't exist
os.makedirs(images_folder, exist_ok=True)
os.makedirs(annotations_folder, exist_ok=True)

# Get all files and separate by type
image_files = []
annotation_files = []

for f in os.listdir(input_folder):
    filepath = os.path.join(input_folder, f)
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        image_files.append(f)
    elif f.lower().endswith(('.xml', '.json', '.txt')):
        annotation_files.append(f)

# Create sets of base filenames (without extensions)
image_basenames = {os.path.splitext(f)[0] for f in image_files}
annotation_basenames = {os.path.splitext(f)[0] for f in annotation_files}

# Find common base filenames (files that have both image and annotation)
common_basenames = image_basenames.intersection(annotation_basenames)

# Convert to list and sort for consistency
common_basenames = sorted(list(common_basenames))

# Limit to 30,000 files
common_basenames = common_basenames[:30000]

print(f"Found {len(common_basenames)} matching image-annotation pairs")

# Create mapping from base names to actual filenames
image_name_map = {}
for img in image_files:
    base = os.path.splitext(img)[0]
    image_name_map[base] = img

annotation_name_map = {}
for ann in annotation_files:
    base = os.path.splitext(ann)[0]
    annotation_name_map[base] = ann

# Copy the matching files
copied_count = 0
for base_name in common_basenames:
    # Copy image
    if base_name in image_name_map:
        img_source = os.path.join(input_folder, image_name_map[base_name])
        img_dest = os.path.join(images_folder, image_name_map[base_name])
        shutil.copy(img_source, img_dest)
    
    # Copy annotation
    if base_name in annotation_name_map:
        ann_source = os.path.join(input_folder, annotation_name_map[base_name])
        ann_dest = os.path.join(annotations_folder, annotation_name_map[base_name])
        shutil.copy(ann_source, ann_dest)
    
    copied_count += 1
    if copied_count % 5000 == 0:
        print(f"Copied {copied_count} files...")

print(f"Successfully copied {copied_count} image-annotation pairs")
print(f"Images saved to: {images_folder}")
print(f"Annotations saved to: {annotations_folder}")

Found 30000 matching image-annotation pairs
Copied 5000 files...
Copied 10000 files...
Copied 15000 files...
Copied 20000 files...
Copied 25000 files...
Copied 30000 files...
Successfully copied 30000 image-annotation pairs
Images saved to: /kaggle/working/images
Annotations saved to: /kaggle/working/annotations


In [ ]:
import os

dataset_path = '/kaggle/working/annotations'

# Find all files
all_files = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        all_files.append(os.path.join(root, f))

# Count all files
print(f"Total number of files: {len(all_files)}")


In [ ]:
# import os
# os.listdir('/content/vehicle_dataset')


In [ ]:
# Step 1: Check hardware type
!nvidia-smi


In [1]:
# Step 2: Check CUDA from Python
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


CUDA available: True
GPU Name: NVIDIA A100-SXM4-40GB


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available (runtime):", torch.cuda.is_available())
print("CUDA built-in (build):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())



PyTorch version: 2.8.0+cu126
CUDA available (runtime): True
CUDA built-in (build): 12.6
cuDNN version: 91002


In [ ]:
# !nvidia-smi



In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.ops as ops
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import math

In [4]:
import os
print(f"Available CPU cores: {os.cpu_count()}")
# Set num_workers = CPU_cores - 2

Available CPU cores: 256


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.ops as ops
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import math



Found 2000 valid image-annotation pairs
Image shape: torch.Size([3, 512, 512])
Number of boxes: 3
Box coordinates: tensor([[0.6594, 0.4287, 0.6755, 0.4500],
        [0.6875, 0.4306, 0.7120, 0.4704],
        [0.4177, 0.1046, 0.6740, 0.7824]])
Labels: tensor([2, 2, 6])
Bounding box test saved to bbox_test.png
Found 2000 valid image-annotation pairs
Class counts: tensor([3.6720e+03, 1.3240e+03, 2.7530e+03, 1.1800e+02, 1.4000e+01, 2.2000e+01,
        7.7800e+02, 8.9000e+01, 5.2900e+02, 5.7000e+01, 3.0000e+00, 1.9000e+01,
        5.7000e+01, 1.0300e+02, 4.7000e+01, 0.0000e+00])
Class weights: tensor([4.3573e-09, 1.2085e-08, 5.8118e-09, 1.3559e-07, 1.1429e-06, 7.2727e-07,
        2.0566e-08, 1.7978e-07, 3.0246e-08, 2.8070e-07, 5.3333e-06, 8.4210e-07,
        2.8070e-07, 1.5534e-07, 3.4043e-07, 1.6000e+01])
Class counts: tensor([3.6720e+03, 1.3240e+03, 2.7530e+03, 1.1800e+02, 1.4000e+01, 2.2000e+01,
        7.7800e+02, 8.9000e+01, 5.2900e+02, 5.7000e+01, 3.0000e+00, 1.9000e+01,
        5.7000

KeyboardInterrupt: 

In [ ]:

# Set device to CUDA if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
# Configuration
NUM_CLASSES = 16  # Adjust based on your classes (0-8 from your data + background)
IMG_SIZE = 512
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 12
K_FOLDS = 3
# Remove this line:
# IMG_SIZE = 300
# Add this instead:


def get_image_size(image_path):
    with Image.open(image_path) as img:
        return img.size  # Returns (width, height)

# Set random seeds for reproducibility
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ==================== VGG BACKBONE ====================
class VGGBackbone(nn.Module):
    def __init__(self):
        super(VGGBackbone, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=1)
        )

        self.conv5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        conv4_3 = self.conv4(x)
        x = self.conv5(conv4_3)
        return conv4_3, x

# ==================== EXTRA FEATURE LAYERS ====================
class ExtraFeatureLayers(nn.Module):
    def __init__(self):
        super(ExtraFeatureLayers, self).__init__()
        self.conv8 = nn.Sequential(
            nn.Conv2d(1024, 256, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

        self.conv9 = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

        self.conv10 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )

        self.conv11 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        features = []
        x = self.conv8(x)
        features.append(x)
        x = self.conv9(x)
        features.append(x)
        x = self.conv10(x)
        features.append(x)
        x = self.conv11(x)
        features.append(x)
        return features

# ==================== PREDICTION HEADS ====================
class PredictionHeads(nn.Module):
    def __init__(self, num_classes):
        super(PredictionHeads, self).__init__()
        self.num_classes = num_classes

        feature_channels = [512, 1024, 512, 256, 256, 256]
        aspect_ratios = [[2], [2, 3], [2, 3], [2, 3], [2], [2]]

        self.num_anchors = []
        for k in range(len(feature_channels)):
            num_anchors = len(aspect_ratios[k]) + 1
            self.num_anchors.append(num_anchors)

        print(f"Number of anchors per feature map: {self.num_anchors}")

        self.loc_layers = nn.ModuleList([
            nn.Conv2d(feature_channels[i], self.num_anchors[i] * 4, kernel_size=3, padding=1)
            for i in range(len(feature_channels))
        ])

        self.conf_layers = nn.ModuleList([
            nn.Conv2d(feature_channels[i], self.num_anchors[i] * num_classes, kernel_size=3, padding=1)
            for i in range(len(feature_channels))
        ])

    def forward(self, features):
        locs = []
        confs = []

        for i, feature in enumerate(features):
            loc = self.loc_layers[i](feature)
            conf = self.conf_layers[i](feature)

            batch_size = loc.size(0)
            loc = loc.permute(0, 2, 3, 1).contiguous().view(batch_size, -1, 4)
            conf = conf.permute(0, 2, 3, 1).contiguous().view(batch_size, -1, self.num_classes)

            locs.append(loc)
            confs.append(conf)

        locs = torch.cat(locs, dim=1)
        confs = torch.cat(confs, dim=1)

        return locs, confs

# ==================== SSD MODEL ====================
class SSD(nn.Module):
    def __init__(self, num_classes):
        super(SSD, self).__init__()
        self.num_classes = num_classes
        self.backbone = VGGBackbone()
        self.extra_layers = ExtraFeatureLayers()
        self.prediction_heads = PredictionHeads(num_classes)
        self.prior_boxes = self.create_prior_boxes()

    def create_prior_boxes(self):
        dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
        with torch.no_grad():
            conv4_3, conv7 = self.backbone(dummy_input)
            extra_features = self.extra_layers(conv7)
            features = [conv4_3, conv7] + extra_features
        
        feature_maps = [f.size(2) for f in features]  # Dynamic calculation
        print(f"Actual feature map sizes: {feature_maps}")
        # feature_maps = [37, 37, 19, 10, 8, 6]
        scales = [0.1, 0.2, 0.375, 0.55, 0.725, 0.9]
        aspect_ratios = [[2], [2, 3], [2, 3], [2, 3], [2], [2]]

        prior_boxes = []

        for k, fmap in enumerate(feature_maps):
            for i in range(fmap):
                for j in range(fmap):
                    cx = (j + 0.5) / fmap
                    cy = (i + 0.5) / fmap

                    for ratio in aspect_ratios[k]:
                        w = scales[k] * math.sqrt(ratio)
                        h = scales[k] / math.sqrt(ratio)
                        prior_boxes.append([cx, cy, w, h])

                    scale_prime = math.sqrt(scales[k] * (scales[k + 1] if k + 1 < len(scales) else 1.0))
                    prior_boxes.append([cx, cy, scale_prime, scale_prime])

        print(f"Total prior boxes generated: {len(prior_boxes)}")
        prior_boxes = torch.FloatTensor(prior_boxes).clamp_(min=0, max=1)
        return prior_boxes

    def forward(self, x):
        conv4_3, conv7 = self.backbone(x)
        extra_features = self.extra_layers(conv7)
        features = [conv4_3, conv7] + extra_features
        locs, confs = self.prediction_heads(features)
        return locs, confs

# ==================== DATASET (FIXED) ====================
class YOLODataset(Dataset):
    def __init__(self, image_dir, annotation_dir, transform=None):
        self.image_dir = image_dir
        self.annotation_dir = annotation_dir
        self.transform = transform

        # FIXED: Get all image files and verify matching annotations
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.PNG', '.JPEG'))]

        self.valid_files = []
        for img_file in self.image_files:
            base_name = os.path.splitext(img_file)[0]
            txt_file = base_name + '.txt'
            if os.path.exists(os.path.join(annotation_dir, txt_file)):
                self.valid_files.append(img_file)
        self.valid_files=self.valid_files[:35000]
        print(f"Found {len(self.valid_files)} valid image-annotation pairs")

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        img_name = self.valid_files[idx]
        img_path = os.path.join(self.image_dir, img_name)

        base_name = os.path.splitext(img_name)[0]
        txt_path = os.path.join(self.annotation_dir, base_name + '.txt')

        image = Image.open(img_path).convert('RGB')

        boxes = []
        labels = []

        with open(txt_path, 'r') as f:
            for line in f:
                data = line.strip().split()
                if len(data) >= 5:
                    class_id = int(data[0])
                    cx = float(data[1])
                    cy = float(data[2])
                    w = float(data[3])
                    h = float(data[4])

                    x1 = cx - w/2
                    y1 = cy - h/2
                    x2 = cx + w/2
                    y2 = cy + h/2

                    boxes.append([x1, y1, x2, y2])
                    labels.append(class_id)

        boxes = torch.FloatTensor(boxes) if boxes else torch.zeros((0, 4))
        labels = torch.LongTensor(labels) if labels else torch.zeros(0, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, boxes, labels

def detection_collate_fn(batch):
    images = []
    boxes = []
    labels = []

    for sample in batch:
        images.append(sample[0])
        boxes.append(sample[1])
        labels.append(sample[2])

    images = torch.stack(images, 0)
    return images, boxes, labels

# ==================== LOSS FUNCTION (FIXED) ====================
class SSDLoss(nn.Module):
    def __init__(self, num_classes, overlap_threshold=0.5, prior_boxes=None):
        super(SSDLoss, self).__init__()
        self.num_classes = num_classes
        self.overlap_threshold = overlap_threshold
        self.prior_boxes = prior_boxes
        # self.class_weights = torch.tensor([
        #     0.5,  # Class 0 - reduce weight (many samples)
        #     0.75,  # Class 1 - reduce weight (many samples)  
        #     0.5,  # Class 2
        #     5.0,  # Class 3
        #     5.0,  # Class 4
        #     5.0,  # Class 5
        #     1.5,  # Class 6
        #     1.5,  # Class 7
        #     1.5,  # Class 8
        #     10.0,  # Class 9
        #     10.0,  # Class 10
        #     10.0,  # Class 11
        #     3.5,  # Class 12 - increase weight (rare)
        #     3.5,  # Class 13 - increase more (very rare)
        #     3.5,  # Class 14 - increase more (very rare)
        #     0.1   # Class 15 (background) - reduce weight
        # ])

    def forward(self, predictions, targets):
        loc_data, conf_data = predictions
        boxes, labels = targets

        batch_size = loc_data.size(0)
        num_priors = loc_data.size(1)

        loc_t = torch.zeros(batch_size, num_priors, 4, device=loc_data.device)
        conf_t = torch.zeros(batch_size, num_priors, dtype=torch.long, device=loc_data.device)

        for idx in range(batch_size):
            truths = boxes[idx]
            labels_truth = labels[idx]
            defaults = self.prior_boxes.to(loc_data.device)

            if truths.numel() == 0:
                conf_t[idx] = 15
                continue

            ious = self.compute_iou(truths, defaults)
            best_prior_overlap, best_prior_idx = ious.max(1)
            best_truth_overlap, best_truth_idx = ious.max(0)

            for j in range(best_prior_idx.size(0)):
                best_prior_idx_j = best_prior_idx[j]
                best_truth_overlap[best_prior_idx_j] = 2
                best_truth_idx[best_prior_idx_j] = j

            matches = truths[best_truth_idx]
            conf = labels_truth[best_truth_idx]
            conf[best_truth_overlap < self.overlap_threshold] = 15

            loc_t[idx] = self.encode(matches, defaults)
            conf_t[idx] = conf

        pos_mask = conf_t < 15
        num_pos = pos_mask.sum(dim=1, keepdim=True)

        # FIXED: Handle case when no positive samples
        if num_pos.sum() == 0:
            return torch.tensor(0.0, device=loc_data.device, requires_grad=True)

        loc_p = loc_data[pos_mask].view(-1, 4)
        loc_t_pos = loc_t[pos_mask].view(-1, 4)
        loss_l = F.smooth_l1_loss(loc_p, loc_t_pos, reduction='sum') / num_pos.sum()

        weight = torch.ones(self.num_classes).to(conf_data.device)
        weight[15] = 0.1  # Reduce background class weight
        # weight = self.class_weights.to(conf_data.device)

        loss_c = F.cross_entropy(conf_data.view(-1, self.num_classes),
                                conf_t.view(-1), weight=weight, reduction='sum') / num_pos.sum()

        return loss_l + loss_c

    def compute_iou(self, box1, box2):
        n = box1.size(0)
        m = box2.size(0)

        # FIXED: Ensure box2 is on the same device as box1
        box2 = box2.to(box1.device)

        box1 = box1.unsqueeze(1).expand(n, m, 4)
        box2 = box2.unsqueeze(0).expand(n, m, 4)

        inter_left = torch.max(box1[..., 0], box2[..., 0])
        inter_top = torch.max(box1[..., 1], box2[..., 1])
        inter_right = torch.min(box1[..., 2], box2[..., 2])
        inter_bottom = torch.min(box1[..., 3], box2[..., 3])

        inter_area = torch.clamp(inter_right - inter_left, min=0) * torch.clamp(inter_bottom - inter_top, min=0)

        area1 = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
        area2 = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
        union_area = area1 + area2 - inter_area

        return inter_area / (union_area + 1e-6)

    def encode(self, matched, priors):
        g_cxcy = (matched[:, :2] + matched[:, 2:]) / 2 - priors[:, :2]
        g_cxcy /= priors[:, 2:]

        g_wh = (matched[:, 2:] - matched[:, :2]) / priors[:, 2:]
        g_wh = torch.log(g_wh + 1e-6)

        return torch.cat([g_cxcy, g_wh], 1)

# ==================== DETECTION UTILITIES ====================
class DetectionUtils:
    @staticmethod
    def decode(loc, priors):
        boxes = torch.cat((
            priors[:, :2] + loc[:, :2] * priors[:, 2:],
            priors[:, 2:] * torch.exp(loc[:, 2:])
        ), 1)

        boxes[:, :2] -= boxes[:, 2:] / 2
        boxes[:, 2:] += boxes[:, :2]

        return boxes


    @staticmethod
    def detect_objects(predictions, model, confidence_threshold=0.6, nms_threshold=0.70):
        loc_data, conf_data = predictions
        batch_size = loc_data.size(0)

        all_detections = []

        for i in range(batch_size):
            decoded_boxes = DetectionUtils.decode(loc_data[i], model.prior_boxes.to(loc_data.device))
            conf_scores = F.softmax(conf_data[i], dim=1)

            detections = []
            all_boxes = []
            all_scores = []
            all_labels = []

            # Collect all detections across all classes
            for cl in range(0, model.num_classes-1):
                conf_mask = conf_scores[:, cl] > confidence_threshold
                if conf_mask.sum() == 0:
                    continue

                boxes = decoded_boxes[conf_mask]
                scores = conf_scores[conf_mask, cl]
                labels = torch.full((scores.size(0),), cl, device=scores.device)

                all_boxes.append(boxes)
                all_scores.append(scores)
                all_labels.append(labels)

            if all_boxes:
                # Combine all detections
                all_boxes = torch.cat(all_boxes, dim=0)
                all_scores = torch.cat(all_scores, dim=0)
                all_labels = torch.cat(all_labels, dim=0)

                # Apply NMS across all classes
                keep = ops.nms(all_boxes, all_scores, nms_threshold)

                for idx in keep:
                    detections.append({
                        'bbox': all_boxes[idx].tolist(),
                        'score': all_scores[idx].item(),
                        'class': all_labels[idx].item()
                    })

            all_detections.append(detections)

        return all_detections

# ==================== VISUALIZATION ====================
def visualize_feature_maps(model, image_tensor, layer_name):
    model.eval()
    feature_maps = {}

    def get_feature(name):
        def hook(model, input, output):
            feature_maps[name] = output.detach()
        return hook

    hooks = []
    for name, layer in model.named_modules():
        if isinstance(layer, nn.Conv2d):
            hooks.append(layer.register_forward_hook(get_feature(name)))

    with torch.no_grad():
        _ = model(image_tensor.unsqueeze(0).to(device))

    for hook in hooks:
        hook.remove()

    # Plot first 16 channels as heatmaps
    if feature_maps:
        first_feature_key = list(feature_maps.keys())[4]  # Get conv4_3
        feature = feature_maps[first_feature_key]

        fig, axes = plt.subplots(4, 4, figsize=(15, 15))
        fig.suptitle(f'Feature Maps (Heatmaps) - {first_feature_key}', fontsize=16)

        for idx, ax in enumerate(axes.flat):
            if idx < feature.size(1):
                im = ax.imshow(feature[0, idx].cpu().numpy(), cmap='viridis')
                ax.set_title(f'Channel {idx}')
                ax.axis('off')
                plt.colorbar(im, ax=ax, fraction=0.046)

        # plt.tight_layout()
        plt.savefig(f'feature_maps_{first_feature_key}.png', dpi=150)
        plt.close()
        print(f"Feature map heatmaps saved")

def plot_performance_metrics(train_losses, val_losses, val_accuracies, fold):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

    ax1.plot(train_losses, label='Train Loss', linewidth=2)
    ax1.plot(val_losses, label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training and Validation Loss - Fold {fold}', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(val_accuracies, label='Val Accuracy', color='green', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title(f'Validation Accuracy - Fold {fold}', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Combined plot
    ax3_twin = ax3.twinx()
    ax3.plot(train_losses, 'b-', label='Train Loss', linewidth=2)
    ax3.plot(val_losses, 'r-', label='Val Loss', linewidth=2)
    ax3_twin.plot(val_accuracies, 'g-', label='Val Accuracy', linewidth=2)
    ax3.set_xlabel('Epoch', fontsize=12)
    ax3.set_ylabel('Loss', fontsize=12)
    ax3_twin.set_ylabel('Accuracy (%)', fontsize=12)
    ax3.set_title(f'Combined Metrics - Fold {fold}', fontsize=14)
    ax3.legend(loc='upper left')
    ax3_twin.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)

    # plt.tight_layout()
    plt.savefig(f'training_metrics_fold_{fold}.png', dpi=150)
    plt.close()
    print(f"Performance metrics saved to training_metrics_fold_{fold}.png")

def visualize_predictions(model, image, boxes, labels, predictions, image_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    image_np = image.permute(1, 2, 0).cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image_np = image_np * std + mean
    image_np = np.clip(image_np, 0, 1)

    ax1.imshow(image_np)
    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box.cpu().numpy() * IMG_SIZE
        width = x2 - x1
        height = y2 - y1
        rect = patches.Rectangle((x1, y1), width, height, linewidth=2,
                               edgecolor='g', facecolor='none')
        ax1.add_patch(rect)
        ax1.text(x1, y1, f'Class {label.item()}', color='g', fontsize=8,
                bbox=dict(facecolor='white', alpha=0.7))
    ax1.set_title('Ground Truth')
    ax1.axis('off')

    ax2.imshow(image_np)
    for detection in predictions:
        bbox = detection['bbox']
        score = detection['score']
        class_id = detection['class']

        x1, y1, x2, y2 = [v * IMG_SIZE for v in bbox]
        width = x2 - x1
        height = y2 - y1
        rect = patches.Rectangle((x1, y1), width, height, linewidth=2,
                               edgecolor='r', facecolor='none')
        ax2.add_patch(rect)
        ax2.text(x1, y1, f'Class {class_id} ({score:.2f})', color='r', fontsize=8,
                bbox=dict(facecolor='white', alpha=0.7))
    ax2.set_title('Predictions')
    ax2.axis('off')

    # plt.tight_layout()
    plt.savefig(f'prediction_{image_name}.png', dpi=150)
    plt.close()
    print(f"Prediction saved to prediction_{image_name}.png")

# ==================== EVALUATION (FIXED) ====================
def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    all_pred_counts = []
    all_true_counts = []

    with torch.no_grad():
        for images, boxes, labels in data_loader:
            images = images.to(device)
            boxes = [b.to(device) for b in boxes]
            labels = [l.to(device) for l in labels]
            loc_pred, conf_pred = model(images)

            loss = criterion((loc_pred, conf_pred), (boxes, labels))
            total_loss += loss.item()

            predictions = DetectionUtils.detect_objects((loc_pred, conf_pred), model)

            for pred, box_list in zip(predictions, boxes):
                all_pred_counts.append(len(pred))
                all_true_counts.append(box_list.size(0))

    avg_loss = total_loss / len(data_loader)

    # FIXED: Calculate metrics properly
    if len(all_pred_counts) > 0:
        all_pred_counts = np.array(all_pred_counts)
        all_true_counts = np.array(all_true_counts)

        mse = mean_squared_error(all_true_counts, all_pred_counts)
        mae = mean_absolute_error(all_true_counts, all_pred_counts)

        if np.var(all_true_counts) > 0:
            r2 = r2_score(all_true_counts, all_pred_counts)
        else:
            r2 = 0.0
    else:
        mse = mae = r2 = 0.0

    return {
        'loss': avg_loss,
        'mse': mse,
        'mae': mae,
        'r2': r2
    }

# ==================== TRAINING FUNCTION ====================
def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs, fold):
    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for images, boxes, labels in train_loader:
            images = images.to(device)
            boxes = [b.to(device) for b in boxes]
            labels = [l.to(device) for l in labels]

            optimizer.zero_grad()
            loc_pred, conf_pred = model(images)
            loss = criterion((loc_pred, conf_pred), (boxes, labels))
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        correct_detections = 0  # FIXED: Initialize here
        total_detections = 0  

        with torch.no_grad():
            for images, boxes, labels in val_loader:
                images = images.to(device)
                boxes = [b.to(device) for b in boxes]
                labels = [l.to(device) for l in labels]

                loc_pred, conf_pred = model(images)
                loss = criterion((loc_pred, conf_pred), (boxes, labels))
                val_loss += loss.item()

                # NEW CODE (replace with this):
                # Calculate detection accuracy using actual detections
                predictions = DetectionUtils.detect_objects((loc_pred, conf_pred), model, confidence_threshold=0.6)
                
                for i, (pred_list, true_boxes, true_labels) in enumerate(zip(predictions, boxes, labels)):
                    if len(true_boxes) > 0 and len(pred_list) > 0:
                        # Simple detection accuracy: count correct class predictions
                        pred_labels = torch.tensor([pred['class'] for pred in pred_list]).to(device)
                        
                        # Count correct class predictions (simplified accuracy)
                        matched = 0
                        for true_label in true_labels:
                            if true_label in pred_labels:
                                matched += 1
                        correct_detections += matched
                        total_detections += len(true_labels)
                
                
               

        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        accuracy = 100 * correct_detections / total_detections if total_detections > 0 else 0
        val_accuracies.append(accuracy)

        print(f'Fold {fold}, Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {accuracy:.2f}%')

    return train_losses, val_losses, val_accuracies

def visualize_sample_predictions(model, data_loader, fold):
    model.eval()
    images, boxes, labels = next(iter(data_loader))

    with torch.no_grad():
        predictions = model(images.to(device))
        detections = DetectionUtils.detect_objects(predictions, model)

    for i in range(min(3, len(images))):
        image_name = f'fold_{fold}_sample_{i}'
        visualize_predictions(model, images[i], boxes[i], labels[i],
                            detections[i], image_name)

def create_prediction_annotations(model, data_loader, output_dir='predictions'):
    os.makedirs(output_dir, exist_ok=True)
    model.eval()

    with torch.no_grad():
        for batch_idx, (images, _, _) in enumerate(data_loader):
            predictions = model(images.to(device))
            detections = DetectionUtils.detect_objects(predictions, model)

            for i, detection in enumerate(detections):
                image_idx = batch_idx * BATCH_SIZE + i
                if image_idx < len(data_loader.dataset):
                    img_name = data_loader.dataset.valid_files[image_idx]
                    base_name = os.path.splitext(img_name)[0]

                    annotation_file = os.path.join(output_dir, base_name + '.txt')

                    with open(annotation_file, 'w') as f:
                        for det in detection:
                            bbox = det['bbox']
                            score = det['score']
                            class_id = det['class']

                            x1, y1, x2, y2 = bbox
                            cx = (x1 + x2) / 2
                            cy = (y1 + y2) / 2
                            w = x2 - x1
                            h = y2 - y1

                            f.write(f'{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} {score:.6f}\n')

    print(f"Prediction annotations saved to '{output_dir}' folder")

class AddGaussianNoise:
    def __init__(self, mean=0., std=0.1):
        self.std = std
        self.mean = mean
        
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean
    
    def __repr__(self):
        return self.__class__.__name__ + '(mean={0}, std={1})'.format(self.mean, self.std)

# ==================== MAIN FUNCTION ====================
def main():
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        # transforms.RandomHorizontalFlip(0.5),
        # transforms.RandomRotation(10),
        
        # 50% chance for blur
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.2),
        
        transforms.ToTensor(),
        
        # 50% chance for noise
        transforms.RandomApply([AddGaussianNoise(0., 0.05)], p=0.2),
        
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # FIXED: Update these paths
    image_dir = r"/home/tsawang/vehicle_images"
    annotation_dir = r"/home/tsawang/annotation"

    dataset = YOLODataset(image_dir, annotation_dir, transform=transform)

    # FIXED: Check if dataset is valid
    if len(dataset) == 0:
        print("ERROR: No valid data found. Please check your image and annotation directories.")
        return

    kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f'\n{"="*60}')
        print(f'Starting Fold {fold + 1}/{K_FOLDS}')
        print(f'{"="*60}')

        train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)
        val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)

        train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=train_subsampler, collate_fn=detection_collate_fn,num_workers=4)
        val_loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=val_subsampler, collate_fn=detection_collate_fn,num_workers=4)

        model = SSD(NUM_CLASSES).to(device)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        criterion = SSDLoss(num_classes=NUM_CLASSES, prior_boxes=model.prior_boxes)

        train_losses, val_losses, val_accuracies = train_model(
            model, train_loader, val_loader, optimizer, criterion, NUM_EPOCHS, fold + 1
        )

        plot_performance_metrics(train_losses, val_losses, val_accuracies, fold + 1)

        eval_metrics = evaluate_model(model, val_loader, criterion)
        fold_results.append(eval_metrics)

        print(f'\nFold {fold + 1} Results:')
        for metric, value in eval_metrics.items():
            print(f'  {metric}: {value:.4f}')

        torch.save(model.state_dict(), f'ssd_model_fold_{fold + 1}.pth')
        print(f'Model saved to ssd_model_fold_{fold + 1}.pth')

        visualize_sample_predictions(model, val_loader, fold + 1)

        if fold == 0:
            sample_image, _, _ = dataset[0]
            visualize_feature_maps(model, sample_image, layer_name="conv4_3")

    # Print overall results
    print("\n" + "="*60)
    print("K-Fold Cross Validation Results Summary")
    print("="*60)

    for metric in fold_results[0].keys():
        values = [result[metric] for result in fold_results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f'{metric.upper()}: {mean_val:.4f} (+/- {std_val:.4f})')

    print("\nCreating prediction annotations...")
    full_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=detection_collate_fn)
    create_prediction_annotations(model, full_loader, output_dir='predictions')

    print("\n" + "="*60)
    print("Training complete!")
    print("="*60)

# ==================== USAGE FUNCTIONS ====================
def load_and_test_model():
    """Load trained model and make predictions"""
    model = SSD(NUM_CLASSES).to(device)
    model.load_state_dict(torch.load('ssd_model_fold_1.pth', map_location=device))
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    image_dir = r"/home/tsawang/vehicle_images"
    annotation_dir = r"/home/tsawang/annotation"

    dataset = YOLODataset(image_dir, annotation_dir, transform=transform)
    data_loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=detection_collate_fn)

    with torch.no_grad():
        for i, (images, boxes, labels) in enumerate(data_loader):
            if i >= 3:
                break

            predictions = model(images.to(device))
            detections = DetectionUtils.detect_objects(predictions, model)

            print(f"Sample {i+1}:")
            print(f"Ground truth boxes: {len(boxes[0])}")
            print(f"Detected objects: {len(detections[0])}")

            for j, detection in enumerate(detections[0]):
                print(f"  Object {j+1}: Class {detection['class']}, Score: {detection['score']:.3f}")

def predict_single_image(image_path, model_path='ssd_model_fold_1.pth', confidence_threshold=0.01):
    """Predict on a single image"""
    model = SSD(NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    image = Image.open(image_path).convert('RGB')
    original_size = image.size
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        loc_pred, conf_pred = model(image_tensor)
        detections = DetectionUtils.detect_objects((loc_pred, conf_pred), model, confidence_threshold=confidence_threshold)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(image)

    for detection in detections[0]:
        bbox = detection['bbox']
        score = detection['score']
        class_id = detection['class']

        x1, y1, x2, y2 = [v * original_size[0] if i % 2 == 0 else v * original_size[1] for i, v in enumerate(bbox)]

        width = x2 - x1
        height = y2 - y1

        rect = patches.Rectangle((x1, y1), width, height, linewidth=2,
                               edgecolor='r', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1, f'Class {class_id} ({score:.2f})', color='r', fontsize=10,
                bbox=dict(facecolor='white', alpha=0.7))

    ax.set_title('Object Detection Results')
    ax.axis('off')
    plt.savefig('single_image_prediction.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Detected {len(detections[0])} objects")
    print("Result saved to single_image_prediction.png")
    return detections[0]
def test_bounding_box_processing():
    """Test if bounding boxes are processed correctly"""
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ])
    
    image_dir = r"/home/tsawang/vehicle_images"
    annotation_dir = r"/home/tsawang/annotation"
    
    dataset = YOLODataset(image_dir, annotation_dir, transform=transform)
    
    if len(dataset) == 0:
        print("ERROR: No data loaded")
        return
    
    # Test first sample
    image, boxes, labels = dataset[0]
    print(f"Image shape: {image.shape}")
    print(f"Number of boxes: {len(boxes)}")
    print(f"Box coordinates: {boxes}")
    print(f"Labels: {labels}")
    
    # Visualize ground truth
    if len(boxes) > 0:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        image_np = image.permute(1, 2, 0).numpy()
        ax.imshow(image_np)
        
        for box in boxes:
            x1, y1, x2, y2 = box.numpy() * IMG_SIZE
            width = x2 - x1
            height = y2 - y1
            rect = patches.Rectangle((x1, y1), width, height, linewidth=2,
                                   edgecolor='g', facecolor='none')
            ax.add_patch(rect)
        
        ax.set_title('Ground Truth Bounding Boxes Test')
        ax.axis('off')
        plt.savefig('bbox_test.png', dpi=150)
        plt.close()
        print("Bounding box test saved to bbox_test.png")

# ==================== RUN TRAINING ====================
if __name__ == '__main__':


# Run this before main()
    test_bounding_box_processing()
    main()

    # Uncomment to test trained model:
    # load_and_test_model()

    # # Uncomment to predict on single image (replace with your image path):
    # results = predict_single_image(r"/home/ksuwint/images (10).jpg")

cuda
Found 35000 valid image-annotation pairs
Image shape: torch.Size([3, 512, 512])
Number of boxes: 3
Box coordinates: tensor([[0.6594, 0.4287, 0.6755, 0.4500],
        [0.6875, 0.4306, 0.7120, 0.4704],
        [0.4177, 0.1046, 0.6740, 0.7824]])
Labels: tensor([2, 2, 6])
Bounding box test saved to bbox_test.png
Found 35000 valid image-annotation pairs

Starting Fold 1/3
Number of anchors per feature map: [2, 3, 3, 3, 2, 2]
Actual feature map sizes: [63, 63, 32, 16, 14, 12]
Total prior boxes generated: 24365
Fold 1, Epoch [1/12], Train Loss: 22.0093, Val Loss: 4.9633, Val Acc: 0.00%
Fold 1, Epoch [2/12], Train Loss: 4.2060, Val Loss: 4.0289, Val Acc: 48.75%
Fold 1, Epoch [3/12], Train Loss: 3.8870, Val Loss: 3.8330, Val Acc: 53.95%
Fold 1, Epoch [4/12], Train Loss: 3.6627, Val Loss: 3.5194, Val Acc: 53.92%
Fold 1, Epoch [5/12], Train Loss: 3.4538, Val Loss: 3.2712, Val Acc: 51.75%
Fold 1, Epoch [6/12], Train Loss: 3.2101, Val Loss: 3.0904, Val Acc: 51.18%
Fold 1, Epoch [7/12], Train L

In [44]:
def predict_single_image(image_path, model_path='ssd_model_fold_2.pth', confidence_threshold=0.6):
    """Predict on a single image"""
    model = SSD(NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    image = Image.open(image_path).convert('RGB')
    original_size = image.size
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        loc_pred, conf_pred = model(image_tensor)
        detections = DetectionUtils.detect_objects((loc_pred, conf_pred), model, confidence_threshold=confidence_threshold)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(image)

    for detection in detections[0]:
        bbox = detection['bbox']
        score = detection['score']
        class_id = detection['class']

        x1, y1, x2, y2 = [v * original_size[0] if i % 2 == 0 else v * original_size[1] for i, v in enumerate(bbox)]

        width = x2 - x1
        height = y2 - y1

        rect = patches.Rectangle((x1, y1), width, height, linewidth=2,
                               edgecolor='r', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1, f'Class {class_id} ({score:.2f})', color='r', fontsize=10,
                bbox=dict(facecolor='white', alpha=0.7))

    ax.set_title('Object Detection Results')
    ax.axis('off')
    plt.savefig('single_image_prediction.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Detected {len(detections[0])} objects")
    print("Result saved to single_image_prediction.png")
    return detections[0]

In [49]:
load_and_test_model()
# results = predict_single_image(r"/home/tsawang/vehicle_images/00C5668BQF6R0VX74H3Y.jpg")
results = predict_single_image(r"/home/tsawang/vehicle_images/0C4LGFISCR0JOE3RXCI5.jpg")

Number of anchors per feature map: [2, 3, 3, 3, 2, 2]
Actual feature map sizes: [63, 63, 32, 16, 14, 12]
Total prior boxes generated: 24365
Found 35000 valid image-annotation pairs
Sample 1:
Ground truth boxes: 5
Detected objects: 0
Sample 2:
Ground truth boxes: 13
Detected objects: 12
  Object 1: Class 0, Score: 0.926
  Object 2: Class 0, Score: 0.903
  Object 3: Class 0, Score: 0.830
  Object 4: Class 2, Score: 0.808
  Object 5: Class 0, Score: 0.789
  Object 6: Class 0, Score: 0.751
  Object 7: Class 2, Score: 0.718
  Object 8: Class 0, Score: 0.683
  Object 9: Class 2, Score: 0.658
  Object 10: Class 0, Score: 0.647
  Object 11: Class 2, Score: 0.641
  Object 12: Class 0, Score: 0.638
Sample 3:
Ground truth boxes: 2
Detected objects: 0
Number of anchors per feature map: [2, 3, 3, 3, 2, 2]
Actual feature map sizes: [63, 63, 32, 16, 14, 12]
Total prior boxes generated: 24365
Detected 10 objects
Result saved to single_image_prediction.png
